# 02 — Feature engineering

## What this notebook does
Builds **spectral indices** (when Landsat-like bands exist) and **scientific interaction** features motivated by hydrology and geochemistry.

## Why it matters
Raw bands and single variables miss **couplings**: e.g. high ET with low rain concentrates solutes; runoff moves sediment and phosphorus.

## Physical interpretation (interactions)

| Feature | Meaning |
|---------|--------|
| **ET / precipitation** | Aridity index proxy: more evap per unit rain → less dilution, higher EC/alkalinity possible |
| **precipitation × runoff** | Wet-period delivery: rain × flow capacity → transport of salts and nutrients |
| **soil moisture × NDMI** | Land moisture status: soil water × vegetation moisture index → catchment wetness before sampling |
| **conductivity / alkalinity** | Ionic strength per unit acid-neutralizing capacity (local interpretation only) |


In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))

from src.features.scientific_interactions import add_scientific_interactions
from src.features.spectral_indices import compute_spectral_indices

RAW = ROOT / "data" / "raw"
for name in ["water_quality_dataset_v1.csv", "water_quality.csv"]:
    csv_path = RAW / name
    if csv_path.exists():
        break
else:
    raise FileNotFoundError("No data/raw/water_quality_dataset_v1.csv or water_quality.csv found.")

df = pd.read_csv(csv_path)
original_cols = set(pd.read_csv(csv_path, nrows=0).columns)
# Spectral indices if B2–B7 present
if set(["B2","B4","B5"]).issubset(df.columns):
    df = compute_spectral_indices(df)
df = add_scientific_interactions(df)
new_cols = [c for c in df.columns if c not in original_cols]
print("Added example columns:", [c for c in new_cols if any(x in c for x in ["ratio","proxy","ndmi","ndvi"])][:15])
df.head()

Added example columns: ['et_precip_ratio', 'precip_runoff_proxy', 'soil_moisture_ndmi', 'cond_alk_ratio']


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,...,cec_clay_ratio,lat,lon,total_alkalinity,electrical_conductance,dissolved_reactive_phosphorus,et_precip_ratio,precip_runoff_proxy,soil_moisture_ndmi,cond_alk_ratio
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,...,1.142857,-28.760833,17.730278,128.912,555.0,10.0,356.949523,2.016244e+06,0.001633,4.305263
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,...,0.931034,-26.861111,28.884722,74.720,162.9,163.0,1.274877,1.131153e+06,0.057167,2.180139
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,...,0.821429,-26.450000,28.085833,89.254,573.0,80.0,1.315633,9.691149e+01,-0.032776,6.419880
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,...,0.884615,-27.671111,27.236944,82.000,203.6,101.0,1.267745,1.921849e+06,0.019653,2.482927
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,...,0.892857,-27.356667,27.286389,56.100,145.1,151.0,1.338668,4.611433e+05,0.061796,2.586453


## Save engineered table (optional)
Use the same row order as your original file when merging with targets.


In [2]:
out = ROOT / "data" / "processed" / "features_engineered.csv"
out.parent.mkdir(parents=True, exist_ok=True)
# df.to_csv(out, index=False)
# print("Saved", out)
